# 2 Byte-Pair Encoding (BPE) Tokenizer

### Problem (unicode1): Understanding Unicode

In [1]:
# (a) '\x00'. There's not a single pre-defined symbol representing character chr(0), so it's represented by the hex representation of its code point
chr(0)

'\x00'

In [2]:
# (b): this is a string whose content is the printed representation
chr(0).__repr__()

"'\\x00'"

In [3]:
# (c) this is the string representation of the value
chr(0)

'\x00'

In [4]:
print(chr(0)) # nothing is printed because there's no symbol corresponding to \x00

 


In [5]:
"this is a test" + chr(0) + "string" # the value of the string, where \x00 is used

'this is a test\x00string'

In [6]:
print("this is a test" + chr(0) + "string") # again, since there's no symbol corresponding to \x00, it's not printed

this is a test string


### Problem (unicode 2): Unicode Encodings

In [7]:
# (a) UTF-8 use much fewer bytes to encode string than UTF-16 and UTF-32

test_string = "你好hello"
print(f"utf-8: {test_string.encode("utf-8")}")
print(f"utf-8 len: {len(test_string.encode("utf-8"))}")
print(f"utf-16 len: {len(test_string.encode("utf-16"))}")
print(f"utf-32 len: {len(test_string.encode("utf-32"))}")

utf-8: b'\xe4\xbd\xa0\xe5\xa5\xbdhello'
utf-8 len: 11
utf-16 len: 16
utf-32 len: 32


In [8]:
# (b) Some unicode characters are represented by multiple bytes. This function treats each byte as a single character, which doesn't handle multi-byte characters correctly.

def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

decode_utf8_bytes_to_str_wrong(b'\xe4\xbd\xa0\xe5\xa5\xbdhello')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe4 in position 0: unexpected end of data

In [10]:
# (c) According to utf-8 spec, byte \xe4 must be followed by a byte whos binary representation starts with prefix bits 10
b'\xe4\xe4'.decode("utf-8")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe4 in position 0: invalid continuation byte

### Problem (train_bpe_tinystories): BPE Training on TinyStories

#### Part (a) 
It took 46 seconds, with 42 seconds on pretokenization and 3 seconds on tokenization. Peak memory usage = 152M. 

The following tokens are the longest in the vocab, each with 15 bytes: "Ġresponsibility", "Ġaccomplishment", "Ġdisappointment". They seem to make sense because they correspond to " responsibility", " accomplishment" and " disappointment" in English.

#### Part (b)

Based on the scalene profiling below, the step of using regex to break documents into tokenizations took the most of time.

![Alt Text](tinystories_profile.png)


### Problem (train_bpe_expts_owt): BPE Training on OpenWebText

#### Part (a)

The longest tokens in the vocabulary are "ÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤÃĥÃĤ" and "----------------------------------------------------------------", each with 64 bytes.

They seem to make sense because the former corresponds to "ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ", which seem to appear in the text as separators/fillers a lot, and the latter is a common separator.

#### Part (b)

Similarities: both are dominated by English words

Contrasts: owt vocab is more diverse, with more unicode phrases (still rare though) and lots of variants of separators

### Problem (tokenizer_experiments): Experiments with tokenizers

#### Part (a)

- TinyStories: 7552 bytes tokenized into 1815 tokens (4.16 bytes/token)
- OpenWebText: 31604 bytes tokenized into 6721 tokens (4.7 bytes/token)

#### Part (b)
- When using TinyStories tokenizer, the OpenWebText sample was tokenized into 9882 tokens, leading to a lower compression ratio of 3.2 bytes/token

#### Part (c)

It took 1.25 seconds to tokenize the OpenWebText sample of 31604 bytes. So, the throughput is 25283 bytes / second.

To tokenize Pile dataset, it'd take 825_000_000_000 / 25283 = 32630621 seconds = 9064 hours = 377 days. Since the tokenizer uses a cache, the actual time taken for processing a large corpus is probably shorter.

In [11]:
seconds = 825_000_000_000 / 25283
hours = seconds / 3600
days = hours / 24
print(seconds)
print(hours)
print(days)

32630621.366135348
9064.061490593152
377.66922877471467


#### Part (d)

The bigger vocab size between the two is 32000, so the token ids are all in the range of [0, 2^16) and hence can fit in a uint16 (2 bytes).

# 3 Transformer Language Model Architecture

### Problem (transformer_accounting): Transformer LM resource accounting

#### Part (a)

Trainable parameters in the transformer LM: ~2 billion

Memory usage: ~8GB

In [10]:
import json

GPT2_XL = {
    "name": "GPT-2 XL",
    "vocab_size": 50257,
    "context_length": 1024,
    "num_layers": 48,
    "d_model": 1600,
    "num_heads": 25,
    "d_ff": 6400,
}

def analyze_params(config: dict) -> None:
    vocab_size = config["vocab_size"]
    context_length = config["context_length"]
    num_layers = config["num_layers"]
    d_model = config["d_model"]
    num_heads = config["num_heads"]
    d_ff = config["d_ff"]
    
    num_params = {
        "token_embedding": vocab_size * d_model,
        "transformer_block": {
            "rms_norm1": d_model,
            "multihead_self_attention": 4 * d_model * d_model,
            "rms_norm2": d_model,
            "SwiGLU": 3 * d_model * d_ff
        },
        "rms_norm_final": d_model,
        "linear_final": d_model * vocab_size,
    }

    total_params = (
        num_params["token_embedding"] + 
        num_layers * sum(num_params["transformer_block"].values()) + 
        num_params["rms_norm_final"] +
        num_params["linear_final"]
    )

    print(f"=== Param analysis for {config["name"]} ===")
    print(json.dumps(num_params, indent=4))
    print(f"Total: {total_params:,}")

analyze_params(GPT2_XL)

=== Param analysis for GPT-2 XL ===
{
    "token_embedding": 80411200,
    "transformer_block": {
        "rms_norm1": 1600,
        "multihead_self_attention": 10240000,
        "rms_norm2": 1600,
        "SwiGLU": 30720000
    },
    "rms_norm_final": 1600,
    "linear_final": 80411200
}
Total: 2,127,057,600


#### Part (b)
Matrix multiplications include the following. Total number of FLOPs required for one input sequence: ~4.5 trillion

In [11]:
def analyze_flops(config: dict) -> None:
    vocab_size = config["vocab_size"]
    context_length = config["context_length"]
    num_layers = config["num_layers"]
    d_model = config["d_model"]
    num_heads = config["num_heads"]
    d_ff = config["d_ff"]
    d_head = d_model // num_heads
    matmuls = {
        "transformer_block": {
            "multihead_self_attention": {
                "q_proj": 2 * context_length * d_model * d_model,
                "k_proj": 2 * context_length * d_model * d_model,
                "v_proj": 2 * context_length * d_model * d_model,
                "attention_scoring": num_heads * (2 * context_length * d_head * context_length),
                "applying_attention_weights": num_heads * (2 * context_length * context_length * d_head),
                "out_proj": 2 * context_length * d_model * d_model,
            },
            "SwiGLU": {
                "swiglu_w1_proj": 2 * context_length * d_model * d_ff,
                "swiglu_w3_proj": 2 * context_length * d_model * d_ff,
                "swiglu_w2_proj": 2 * context_length * d_model * d_ff,
            }
        },
        "final_linear": 2 * context_length * vocab_size * d_model
    }

    mhsa_flops = num_layers * sum(matmuls["transformer_block"]["multihead_self_attention"].values())
    swiglu_flops = num_layers * sum(matmuls["transformer_block"]["SwiGLU"].values())
    fl_flops = matmuls["final_linear"]
    total_flops = mhsa_flops + swiglu_flops + fl_flops

    print(f"=== Flops analysis for {config["name"]} ===")
    print(json.dumps(matmuls, indent=4))
    print(f"Total flops: {total_flops:,}")
    print(f"- MultiheadSelfAttention flops: {mhsa_flops:,} ({mhsa_flops/total_flops:.2%})")
    print(f"- SwiGLU flops: {swiglu_flops:,} ({swiglu_flops/total_flops:.2%})")
    print(f"- Final linear flops: {fl_flops:,} ({fl_flops/total_flops:.2%})")

analyze_flops(GPT2_XL)

=== Flops analysis for GPT-2 XL ===
{
    "transformer_block": {
        "multihead_self_attention": {
            "q_proj": 5242880000,
            "k_proj": 5242880000,
            "v_proj": 5242880000,
            "attention_scoring": 3355443200,
            "applying_attention_weights": 3355443200,
            "out_proj": 5242880000
        },
        "SwiGLU": {
            "swiglu_w1_proj": 20971520000,
            "swiglu_w3_proj": 20971520000,
            "swiglu_w2_proj": 20971520000
        }
    },
    "final_linear": 164682137600
}
Total flops: 4,513,336,524,800
- MultiheadSelfAttention flops: 1,328,755,507,200 (29.44%)
- SwiGLU flops: 3,019,898,880,000 (66.91%)
- Final linear flops: 164,682,137,600 (3.65%)


#### Part (c)

SwiGLU requires the most flops

#### Part (d)

As the size of the model increases, SwiGLU becomes more and more dominant, and final linear becomes almost negligbile.

In [12]:
GPT2_small = GPT2_XL.copy()
GPT2_small["name"] = "GPT-2 small"
GPT2_small["d_model"] = 768
GPT2_small["d_ff"] = int(GPT2_small["d_model"] * 8 / 3 / 64) * 64
GPT2_small["num_layers"] = 12
GPT2_small["num_heads"] = 12
analyze_flops(GPT2_small)

=== Flops analysis for GPT-2 small ===
{
    "transformer_block": {
        "multihead_self_attention": {
            "q_proj": 1207959552,
            "k_proj": 1207959552,
            "v_proj": 1207959552,
            "attention_scoring": 1610612736,
            "applying_attention_weights": 1610612736,
            "out_proj": 1207959552
        },
        "SwiGLU": {
            "swiglu_w1_proj": 3221225472,
            "swiglu_w3_proj": 3221225472,
            "swiglu_w2_proj": 3221225472
        }
    },
    "final_linear": 79047426048
}
Total flops: 291,648,307,200
- MultiheadSelfAttention flops: 96,636,764,160 (33.13%)
- SwiGLU flops: 115,964,116,992 (39.76%)
- Final linear flops: 79,047,426,048 (27.10%)


In [14]:
GPT2_medium = GPT2_XL.copy()
GPT2_medium["name"] = "GPT-2 medium"
GPT2_medium["d_model"] = 1024
GPT2_medium["d_ff"] = int(GPT2_medium["d_model"] * 8 / 3 / 64) * 64
GPT2_medium["num_layers"] = 24
GPT2_medium["num_heads"] = 16
analyze_flops(GPT2_medium)

=== Flops analysis for GPT-2 medium ===
{
    "transformer_block": {
        "multihead_self_attention": {
            "q_proj": 2147483648,
            "k_proj": 2147483648,
            "v_proj": 2147483648,
            "attention_scoring": 2147483648,
            "applying_attention_weights": 2147483648,
            "out_proj": 2147483648
        },
        "SwiGLU": {
            "swiglu_w1_proj": 5637144576,
            "swiglu_w3_proj": 5637144576,
            "swiglu_w2_proj": 5637144576
        }
    },
    "final_linear": 105396568064
}
Total flops: 820,508,622,848
- MultiheadSelfAttention flops: 309,237,645,312 (37.69%)
- SwiGLU flops: 405,874,409,472 (49.47%)
- Final linear flops: 105,396,568,064 (12.85%)


In [15]:
GPT2_large = GPT2_XL.copy()
GPT2_large["name"] = "GPT-2 large"
GPT2_large["d_model"] = 1280
GPT2_large["d_ff"] = int(GPT2_large["d_model"] * 8 / 3 / 64) * 64
GPT2_large["num_layers"] = 36
GPT2_large["num_heads"] = 20
analyze_flops(GPT2_large)

=== Flops analysis for GPT-2 large ===
{
    "transformer_block": {
        "multihead_self_attention": {
            "q_proj": 3355443200,
            "k_proj": 3355443200,
            "v_proj": 3355443200,
            "attention_scoring": 2684354560,
            "applying_attention_weights": 2684354560,
            "out_proj": 3355443200
        },
        "SwiGLU": {
            "swiglu_w1_proj": 8891924480,
            "swiglu_w3_proj": 8891924480,
            "swiglu_w2_proj": 8891924480
        }
    },
    "final_linear": 131745710080
}
Total flops: 1,768,530,903,040
- MultiheadSelfAttention flops: 676,457,349,120 (38.25%)
- SwiGLU flops: 960,327,843,840 (54.30%)
- Final linear flops: 131,745,710,080 (7.45%)


#### Part (e)

Total FLOPs increases to 150 trillion from 4 trillion. Now multihead self attention takes a much bigger proportion of total FLOPs.

In [16]:
GPT2_XL_long_context = GPT2_XL.copy()
GPT2_XL_long_context["context_length"] = 16384
analyze_flops(GPT2_XL_long_context)

=== Flops analysis for GPT-2 XL ===
{
    "transformer_block": {
        "multihead_self_attention": {
            "q_proj": 83886080000,
            "k_proj": 83886080000,
            "v_proj": 83886080000,
            "attention_scoring": 858993459200,
            "applying_attention_weights": 858993459200,
            "out_proj": 83886080000
        },
        "SwiGLU": {
            "swiglu_w1_proj": 335544320000,
            "swiglu_w3_proj": 335544320000,
            "swiglu_w2_proj": 335544320000
        }
    },
    "final_linear": 2634914201600
}
Total flops: 149,522,795,724,800
- MultiheadSelfAttention flops: 98,569,499,443,200 (65.92%)
- SwiGLU flops: 48,318,382,080,000 (32.32%)
- Final linear flops: 2,634,914,201,600 (1.76%)


# 4 Training a Transformer LM

### Problem (learning_rate_tuning): Tuning the learning rate

Loss decays faster for lr=10 than lr=1, and with lr=1e2 and lr=1e3, it diverges.

In [17]:
from collections.abc import Callable, Iterable
from typing import Optional
import torch
import math

class SGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr}
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"] # Get the learning rate.
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p] # Get state associated with p.
                t = state.get("t", 0) # Get iteration number from the state, or initial value.
                grad = p.grad.data # Get the gradient of loss with respect to p.
                p.data -= lr / math.sqrt(t + 1) * grad # Update weight tensor in-place.
                state["t"] = t + 1 # Increment iteration number.
        return loss

In [18]:
def train(lr, n_iter):
    torch.manual_seed(0)
    weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
    opt = SGD([weights], lr=lr)

    for t in range(n_iter):
        opt.zero_grad() # Reset the gradients for all learnable parameters.
        loss = (weights**2).mean() # Compute a scalar loss value.
        print(loss.cpu().item())
        loss.backward() # Run backward pass, which computes gradients.
        opt.step() # Run optimizer step

In [19]:
train(lr=1, n_iter=100)

26.271406173706055
25.231056213378906
24.5224609375
23.959409713745117
23.482616424560547
23.064424514770508
22.689321517944336
22.34758758544922
22.032663345336914
21.7398738861084
21.46575355529785
21.2076473236084
20.963468551635742
20.731544494628906
20.510507583618164
20.299222946166992
20.096738815307617
19.902244567871094
19.715045928955078
19.534543991088867
19.360212326049805
19.191591262817383
19.028274536132812
18.869897842407227
18.716140747070312
18.56671142578125
18.421348571777344
18.279813766479492
18.141891479492188
18.007387161254883
17.87611961364746
17.7479248046875
17.622650146484375
17.500154495239258
17.380311965942383
17.262998580932617
17.148101806640625
17.0355224609375
16.925161361694336
16.816925048828125
16.710735321044922
16.606508255004883
16.504167556762695
16.403648376464844
16.30487823486328
16.207799911499023
16.11235237121582
16.018478393554688
15.926130294799805
15.835253715515137
15.745802879333496
15.657732009887695
15.571001052856445
15.485564231

In [20]:
train(lr=1e1, n_iter=10)

26.271406173706055
16.81369972229004
12.394342422485352
9.697249412536621
7.854771614074707
6.512505531311035
5.492434501647949
4.693441867828369
4.053155899047852
3.530749559402466


In [21]:
train(lr=1e2, n_iter=10)

26.271406173706055
26.271404266357422
4.507460117340088
0.10787364840507507
1.1429225992050108e-16
1.2738581433316626e-18
4.289528597092135e-20
2.5553018721537346e-21
2.192103091734976e-22
2.4356700493370243e-23


In [22]:
train(lr=1e3, n_iter=10)

26.271406173706055
9483.9765625
1638032.0
182213552.0
14759298048.0
931480731648.0
47819176017920.0
2057385568894976.0
7.583084306654822e+16
2.4350125357331907e+18


### Problem (adamwAccounting): Resource accounting for training with AdamW

#### Part (a)

- Parameters: 2 * vocab_size * d_model + (2*num_layers+1) * d_model + 16 * num_layers * d_model^2
    - Embedding: vocab_size * d_model
    - TransformerBlock: num_layers * (2 * d_model + 16 * d_model^2)
        - RMSNorm1: d_model
        - MHA: 4 * d_model * d_model
        - RMSNorm2: d_model
        - SwiGLU: 3 * d_model * d_ff
    - Final RMSNorm: d_model
    - Output Embedding: vocab_size * d_model
- Activations: (num_layers * (16\*d_model + 2\*num_heads\*context_length) + d_model + 2\*vocab_size) * context_length * batch_size
    - TransformerBlock: num_layers * (16\*d_model + 2\*num_heads\*context_length) * context_length
        - RMSNorm1: d_model * context_length
        - QKV: d_model * context_length * 3
        - QK^T: context_length * context_length * num_heads
        - softmax: context_length * context_length * num_heads
        - weighted sum of values: d_model * context_length
        - MHA output: d_model * context_length
        - RMSNorm2: d_model * context_length
        - W1 matrix multiply: d_ff * context_length
        - SiLu: d_ff * context_length
        - W2 matrix multiply: d_model * context_length
    - Final RMSNorm: d_model * context_length
    - Output Embedding: vocab_size * context_length
    - Cross_enrtropy: vocab_size * context_length
- Gradients: same as parameters
- Optimizer states: 2 * parameters

Total: (4 * (2 * vocab_size * d_model + (2*num_layers+1) * d_model + 16 * num_layers * d_model^2) + (num_layers * (16\*d_model + 2\*num_heads\*context_length) + d_model + 2\*vocab_size) * context_length * batch_size) * 4


#### Part (b)

Memory usage = 15,517,753,344 * batch_size + 34,032,921,600

Maximum batch size = 2

In [25]:
vocab_size = 50257
context_length = 1024
num_layers = 48
d_model = 1600
num_heads = 25
d_ff = 6400

n_parameters = (2 * vocab_size * d_model + (2*num_layers+1) * d_model + 16 * num_layers * d_model**2)
n_activations_per_sample = (num_layers * (16*d_model + 2*num_heads*context_length) + d_model + 2*vocab_size) * context_length

non_activation_mem = 4 * 4 * n_parameters
activation_mem = 4 * n_activations_per_sample
memory_limit = 80 * 10**9

max_batch_size = (memory_limit - non_activation_mem) // activation_mem

print(f"n_parameters = {n_parameters:,}")
print(f"Non-activation: {non_activation_mem:,}")
print(f"Activations: {activation_mem:,}")
print(f"Max batch size: {max_batch_size}")

n_parameters = 2,127,057,600
Non-activation: 34,032,921,600
Activations: 15,517,753,344
Max batch size: 2


#### Part (c)

Flops for one step of AdamW = 3 * parameters (updating m) + 3 * parameters (updating v) + 5 * parameters (updating parameters) + 2 * parameters (weight decay) = 13 * parameters

#### Part (d)

It takes about 6.5 days.

In [29]:
forward_pass_flops = 4_513_336_524_800 # from previous calculations
backward_pass_flops = 2 * forward_pass_flops
optimizer_flops = 13 * 2_127_057_600

total_flops_per_step = forward_pass_flops + backward_pass_flops + optimizer_flops
MFU = 19.5 * 10**12
effective_MFU = 0.5 * MFU

secs_per_step = total_flops_per_step / effective_MFU
days_for_400k_steps = secs_per_step * 400_000 / (60 * 60 * 24)
print(days_for_400k_steps)

6.442384294017094


# 7 Experiments

### Problem (learning_rate): Tune the learning rate


#### Part (a)
Swept over 1e-4, 1e-3, 1e-2, 1e-1. Both 1e-3 and 1e-2 achieved a validation loss < 1.45.

#### Part (b)
lr=1e-1 was divergent.

![lr](tinystories_lr.png)


# Problem (batch_size_experiment): Batch size variations

Sweapt over: 1, 64, 128, 256, 512, all trained on the same number of tokens by adjusting training steps.

Batch size of 128/256 achieved the best validation loss of 1.33. A batch size of 1 would take more than a day to train so it was terminated early. Batch size of 64 and 512 achieved a validation loss of 1.40, slightly higher than the best but still less than the 1.45 benchmark.

Validation loss didn't decrease monotonically as batch size increased.

![batch size](tinystories_batch_size.png)

### Problem (generate): Generate text

Once upon a time, there was a little girl named Lily. She loved to play with her toys and have fun. One day, she found a big, red ball in her room. She was very happy and wanted to play with it.

Lily tried to squeeze the ball, but it was too big. She tried to squeeze it, but it was too big. She felt sad and didn't know what to do. Then, her mom came into the room and saw Lily was sad.

Her mom said, "Don't worry, Lily. I will help you." She took the big, red ball and squeezed it. The ball started to roll, and Lily was happy again. She played with the big, red ball all day long.